### Basic TensorFlow API

In [90]:
# Install TensorFlow if not already installed
%pip install tensorflow

Note: you may need to restart the kernel to use updated packages.


In [91]:
import tensorflow as tf 

In [92]:
daily_sales_stats = [21, 22, -108, 31, -1, 32, 34, 31]

### Simple Tf dataSet from Python 

In [93]:
tf_set = tf.data.Dataset.from_tensor_slices(daily_sales_stats)
tf_set

## To inspect contents an different way to inspect

for sales in tf_set:
    print(sales)

for sales in tf_set.take(3):
    print(sales.numpy())

tf.Tensor(21, shape=(), dtype=int32)
tf.Tensor(22, shape=(), dtype=int32)
tf.Tensor(-108, shape=(), dtype=int32)
tf.Tensor(31, shape=(), dtype=int32)
tf.Tensor(-1, shape=(), dtype=int32)
tf.Tensor(32, shape=(), dtype=int32)
tf.Tensor(34, shape=(), dtype=int32)
tf.Tensor(31, shape=(), dtype=int32)
21
22
-108


2025-09-15 12:51:42.449428: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


### How To filter data-content

In [94]:
filtered_tf = tf_set.filter(lambda x : x >= 0)

### Applying Linear transformations

In [95]:
mapped_tf = filtered_tf.map(lambda x : x*72)

### How to Shuffle Data 

The shuffle recieves an argument which is the "shuffle buffer size" in which it will apply a Leave One out Approach to shuffling on the buffer size

In [96]:
shuffled_tf = filtered_tf.shuffle(3)

for sales in shuffled_tf.as_numpy_iterator():
    print(sales)

21
32
22
31
31
34


### Creating DataSet Batches 

This abstract function is helpful when we're dealing in larger size data in which our RAM may not have enough memory to run our model.

In [97]:
for sales_batch in shuffled_tf.batch(4):
    print(sales_batch.numpy())

[21 32 22 31]
[34 31]


### Using TensorFLow image Pipiline 

In [98]:
images_ds = tf.data.Dataset.list_files('/Users/yossipartouche/Desktop/University/Year 3/Machine_Learning/ML_Lessons/TensorFlow/dog images - Google Search/*.jpg', shuffle=False)

for file in images_ds.take(3):
    print(file.numpy())

b'/Users/yossipartouche/Desktop/University/Year 3/Machine_Learning/ML_Lessons/TensorFlow/dog images - Google Search/05DOG-AGGRESSION-qhgl-mediumSquareAt3X.jpg'
b'/Users/yossipartouche/Desktop/University/Year 3/Machine_Learning/ML_Lessons/TensorFlow/dog images - Google Search/1024px-Dog_Breeds.jpg'
b'/Users/yossipartouche/Desktop/University/Year 3/Machine_Learning/ML_Lessons/TensorFlow/dog images - Google Search/10HS-DOG-EMOTIONS-jtzp-mediumSquareAt3X.jpg'


In [99]:
class_names = ["cat", "dog"]

In [100]:
image_count = len(images_ds)
image_count


174

In [101]:
train_size = int(image_count*0.8)

train_ds = images_ds.take(train_size)
test_ds = images_ds.skip(train_size)

In [102]:
len(train_ds)


139

In [103]:
s = '/Users/yossipartouche/Desktop/University/Year 3/Machine_Learning/ML_Lessons/TensorFlow/dog images - Google Search/1024px-Dog_Breeds.jpg'

s.split("/")[-1]


def get_label(file_path):
    import os
    return tf.strings.split(file_path, os.path.sep)[-1]

In [104]:
def process_image(file_path):
    label = get_label(file_path)
    img = tf.io.read_file(file_path)

    img_decoded = tf.image.decode_jpeg(img)
    img_resized = tf.image.resize(img_decoded, [128,128])

    return img_resized, label 

In [105]:
for t in train_ds.take(4):
    print(t.numpy())

b'/Users/yossipartouche/Desktop/University/Year 3/Machine_Learning/ML_Lessons/TensorFlow/dog images - Google Search/05DOG-AGGRESSION-qhgl-mediumSquareAt3X.jpg'
b'/Users/yossipartouche/Desktop/University/Year 3/Machine_Learning/ML_Lessons/TensorFlow/dog images - Google Search/1024px-Dog_Breeds.jpg'
b'/Users/yossipartouche/Desktop/University/Year 3/Machine_Learning/ML_Lessons/TensorFlow/dog images - Google Search/10HS-DOG-EMOTIONS-jtzp-mediumSquareAt3X.jpg'
b'/Users/yossipartouche/Desktop/University/Year 3/Machine_Learning/ML_Lessons/TensorFlow/dog images - Google Search/1200px-Labrador_Retriever_portrait.jpg'


In [106]:
for img, label in train_ds.map(process_image).take(3):
    print("Image", img)
    print("Label", label)


Image tf.Tensor(
[[[151.       151.       151.      ]
  [151.       151.       151.      ]
  [152.       152.       152.      ]
  ...
  [ 33.46875   37.46875   38.46875 ]
  [ 27.8125    28.8125    30.8125  ]
  [ 13.999023  17.999023  18.999023]]

 [[155.74707  155.74707  155.74707 ]
  [156.1875   156.1875   156.1875  ]
  [154.59375  154.59375  154.59375 ]
  ...
  [ 33.84375   37.84375   38.84375 ]
  [ 29.375     33.375     34.375   ]
  [ 28.3125    32.3125    32.5     ]]

 [[159.30762  159.30762  159.30762 ]
  [159.86035  160.86035  155.86035 ]
  [157.77441  157.77441  155.77441 ]
  ...
  [ 40.34375   41.34375   43.34375 ]
  [ 40.2666    44.2666    45.2666  ]
  [ 43.        47.        48.      ]]

 ...

 [[127.25     126.25     121.25    ]
  [108.918945 107.918945 102.918945]
  [120.51367  119.51367  114.51367 ]
  ...
  [117.69141  119.69141  118.69141 ]
  [ 76.1709    80.1709    79.1709  ]
  [ 76.6084    80.6084    79.6084  ]]

 [[ 75.99414   77.80664   71.40039 ]
  [113.67383  113.67

In [111]:
def scale(image, label):
    return image/255, label 

In [112]:
train_processed = train_ds.map(process_image)
train_dsScaled = train_processed.map(scale)
for image, label in train_dsScaled.take(5):
    print ("***Image: ", image.numpy()[0][0])
    print("***Label: ", label.numpy())

***Image:  [0.5921569 0.5921569 0.5921569]
***Label:  b'05DOG-AGGRESSION-qhgl-mediumSquareAt3X.jpg'
***Image:  [0.16862746 0.30588236 0.1882353 ]
***Label:  b'1024px-Dog_Breeds.jpg'
***Image:  [0.39216068 0.38823912 0.40784696]
***Label:  b'10HS-DOG-EMOTIONS-jtzp-mediumSquareAt3X.jpg'
***Image:  [0.84705883 0.827451   0.8509804 ]
***Label:  b'1200px-Labrador_Retriever_portrait.jpg'
***Image:  [0.7416667 0.7607843 0.7857843]
***Label:  b'1FD432BD-1DC9-4EE3-B2F3494EBAE7DCAE_source.jpg'
***Image:  [0.7416667 0.7607843 0.7857843]
***Label:  b'1FD432BD-1DC9-4EE3-B2F3494EBAE7DCAE_source.jpg'


## Additional TensorFlow Exercise